# An Introduction to the Black–Litterman in Python

## Introduction

### Background and Theory

The Black–Litterman asset allocation model \cite{black1992global}, \cite{he1999intuition} provides a structured way to blend an investor’s subjective return views with the return expectations implied by market equilibrium. Although it began as an internal Goldman Sachs working paper, it has since been widely adopted by both practitioners and academics.

Conceptually, Black–Litterman is a Bayesian shrinkage approach: market-implied expected returns act as a prior, and investor views provide additional information. Combining the two produces posterior estimates of expected returns and risk, typically denoted by $\mu^{BL}$ and $\Sigma^{BL}$.

A major practical benefit is improved stability in portfolio optimization. Classic Markowitz optimization is notoriously sensitive to errors in estimated expected returns and covariances—small input changes can lead to extreme, unstable weights that drift far from the market portfolio (e.g. \cite{chopra1993effect}, \cite{michaud1989markowitz}). Because Black–Litterman’s posterior parameters are anchored to the market portfolio and adjusted only to the extent supported by the investor’s confidence in their views, the resulting optimized portfolios tend to deviate from the market in a controlled, interpretable way. In the limiting case with no views (and appropriate parameter settings), the Black–Litterman-implied Markowitz solution recovers the market equilibrium portfolio.



### Neutral prior via reverse engineering (Black–Litterman intuition)

**The neutral prior distribution is obtained by reverse engineering**, assuming the market (or benchmark) portfolio is the **optimal portfolio**.

#### Engineering (forward)
Given expected returns and covariances, portfolio optimization produces portfolio weights:

\begin{equation}
(\mu_i,\ \sigma_{ij})_{i,j=1,\ldots,N}
\ \xrightarrow{\ \text{portfolio optimization}\ }\
(w_i^*)_{i=1,\ldots,N}
\end{equation}

#### Reverse engineering (implied equilibrium returns)
Given the covariance matrix and benchmark (market) weights, reverse portfolio optimization produces **implied** expected returns:

\begin{equation}
(\sigma_{ij},\ w_i^{\text{benchmark}})_{i,j=1,\ldots,N}
\ \xrightarrow{\ \text{reverse portfolio optimization}\ }\
\Pi = (\mu_i^{\text{implied}})_{i=1,\ldots,N}
\end{equation}


### The Black–Litterman formulas

Assume **$N$** assets and **$K$** investor views. Black–Litterman combines two input blocks:

#### Market (prior) inputs


\begin{array}{ll}
w & \text{Equilibrium market weights } (N\times 1)\\
\Sigma & \text{Asset covariance matrix } (N\times N)\\
R_f & \text{Risk-free rate}\\
\delta & \text{Risk aversion parameter}\\
\tau & \text{Scalar controlling uncertainty in the prior}
\end{array}


* **$\delta$** is often set by convention (common ballpark values are around 2–3), or estimated as a “market price of risk” proxy:
\begin{equation}
  \delta \approx \frac{\mu_M}{\sigma_M^2}
\end{equation}
  using a broad market index to estimate $\mu_M$ and $\sigma_M^2$.

* **$\tau$** is frequently treated inconsistently in practice. A common rule of thumb is:
\begin{equation}
  \tau = \frac{1}{T}
\end{equation}
  where $T$ is the number of return observations used to estimate $\Sigma$. (Example: 5 years of monthly data $\Rightarrow T=60$, so $\tau\approx 1/60 \approx 0.02$.)

#### View (likelihood) inputs


\begin{array}{ll}
Q & \text{View returns vector } (K\times 1)\\
P & \text{Pick / projection matrix } (K\times N)\\
\Omega & \text{View uncertainty covariance } (K\times K)
\end{array}


**How $P$ and $Q$ encode views**

* **Absolute view (on asset $i$)**: set $Q_k$ to the expected return of asset $i$, and in row $k$ of $P$, set $P_{k i}=1$ and all other entries to $0$.

* **Relative view (asset $j$ outperforms $i$ by $d$)**: set $Q_k=d$, and in row $k$ of $P$, set $P_{k i}=-1$, $P_{k j}=+1$, others $0$.

**Setting $\Omega$**

$\Omega$ can be supplied directly (e.g., from stated confidence, model residual variance, market data, etc.). A common default is to assume independent views and use the diagonal of:
\begin{equation}
P \tau \Sigma P^\top
\end{equation}
i.e.
\begin{equation}
\Omega = \operatorname{diag}\big(P\tau\Sigma P^\top\big)
\end{equation}

#### The Master Formula

The first step is a *reverse-optimization* step that infers the implied returns vector $\pi$ from the equilibrium weights $w$:

\begin{equation}
\pi = \delta \Sigma w
\end{equation}

Next, the posterior returns and covariances are obtained from the *Black–Litterman master formula*:

\begin{equation}
\mu^{BL} =
\left[(\tau\Sigma)^{-1} + P^{T}\Omega^{-1}P\right]^{-1}
\left[(\tau\Sigma)^{-1}\pi + P^{T}\Omega^{-1}Q\right]
\end{equation}

\begin{equation}
\Sigma^{BL} = \Sigma +
\left[(\tau\Sigma)^{-1} + P^{T}\Omega^{-1}P\right]^{-1}
\end{equation}

#### Inverting $\Omega$

The Black–Litterman master formulas involve the term $\Omega^{-1}$. In practice, $\Omega$ can be singular (non-invertible), which makes the formulas hard to implement as written. A standard workaround is to use algebraically equivalent expressions that avoid explicitly inverting $\Omega$, and are often more numerically stable (see \cite{walters2011black} for derivations).

\begin{equation}
\mu^{BL} = \pi + \tau \Sigma P^T\left[(P \tau \Sigma P^T) + \Omega\right]^{-1}\left[Q - P\pi\right]
\end{equation}

\begin{equation}
\Sigma^{BL} = \Sigma + \tau \Sigma - \tau \Sigma P^T\left(P \tau \Sigma P^T + \Omega\right)^{-1} P \tau \Sigma
\end{equation}


### Flavors of Black–Litterman

The original Black–Litterman framework has accumulated many modifications and extensions over time (see \cite{walters2011black} for a detailed survey). As a result, “Black–Litterman” can refer to slightly different models in different implementations.

Here I follow the terminology in \cite{walters2011black}, which groups implementations into two broad categories:

* **Reference Model:** the original approach associated with \cite{black1992global} and \cite{he1999intuition}.
* **Alternative formulations:** popular variants such as \cite{satchell2000demystification} and Meucci’s framework (e.g. \cite{meucci2005beyond}, \cite{meucci2009enhancing}, \cite{meucci2012fully}), where the $\tau$ parameter is effectively removed—either by setting $\tau=1$ or absorbing it into $\Omega$.

In this notebook, I focus exclusively on the **Reference Model** as described in \cite{black1992global} and \cite{he1999intuition}, and I do not implement Meucci-style extensions.

### Implementation Overview

The notebook proceeds in three stages:

1. **Build the Black–Litterman procedure in Python.**
   I implement the model step-by-step and annotate the code to explain each component.

2. **Validation against \cite{he1999intuition}.**
   I use the implementation to reproduce the results in \cite{he1999intuition} exactly, as a correctness check.

3. **Application and evaluation.**
   I then apply Black–Litterman to a Fama–French 6-portfolio allocation setting. Along the way, I test:

   * absolute vs. relative views,
   * multiple prediction-based view construction approaches (seven strategies),
   * backtests over time with portfolio metrics,
   * comparisons against naive mean–variance optimization that directly uses predicted returns/covariances,
   * and the implications for transaction costs.


## Annotated Implementation of Black–Litterman

### The Code

The Black–Litterman procedure is implemented in Python in the function `bl`. Before writing the body of `bl`, we will define a few helper functions to keep the implementation clean and easier to follow.

NumPy treats a column vector differently from a 1-D array. To avoid shape bugs and to work consistently with column vectors, the helper function below accepts either a 1-D NumPy array or a one-column matrix and returns the data as a column vector. We will call this function `as_colvec`.


In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import inv

In [2]:
def as_colvec(x):
    if (x.ndim == 2):
        return x
    else:
        return np.expand_dims(x, axis=1)

In [3]:
np.arange(4)

array([0, 1, 2, 3])

In [4]:
as_colvec(np.arange(4))

array([[0],
       [1],
       [2],
       [3]])

Recall that the first step in the Black–Litterman procedure is to reverse engineer the implied returns vector $\pi$ from the equilibrium portfolio weights $w$:

\begin{equation}
\pi = \delta \Sigma w
\end{equation}

This is computed by the following code:


In [5]:
def implied_returns(delta, sigma, w):
    """
    Compute implied (equilibrium) expected returns via reverse optimization.

    Inputs:
        delta: Risk aversion coefficient (scalar)
        sigma: Variance–covariance matrix (N x N), DataFrame
        w: Benchmark/market portfolio weights (N x 1), Series

    Returns:
        ir: Implied expected returns (N x 1), Series
    """
    ir = (delta * (sigma @ w)).squeeze()  # squeeze: convert 1-col result to Series
    ir.name = "Implied Returns"
    return ir

As noted earlier, \cite{he1999intuition} suggests a convenient default when the investor does not explicitly specify view uncertainty in the $\Omega$ matrix: assume each view’s uncertainty is proportional to the variance implied by the prior.

Specifically:

\begin{equation}
\Omega = \mathrm{diag}\left(P (\tau \Sigma) P^T\right)
\end{equation}

This is implemented in Python as:


In [6]:
# Assume Omega is proportional to the variance implied by the prior
def proportional_prior(sigma, tau, p):
    """
    He–Litterman default for view uncertainty (Omega).

    Inputs:
        sigma: Covariance matrix (N x N), DataFrame
        tau: Scalar prior-uncertainty parameter
        p: Pick/projection matrix (K x N), DataFrame

    Returns:
        Omega: View-uncertainty matrix (K x K), DataFrame (diagonal)
    """
    helit_omega = p @ (tau * sigma) @ p.T
    # Keep only the diagonal (assume independent views)
    return pd.DataFrame(
        np.diag(np.diag(helit_omega.values)), # np.diag(np.diag(...)) keeps only the diagonal (zeros out off-diagonal terms)
        index=p.index,
        columns=p.index,
    )

We use this function to compute the posterior expected returns as follows:

In [7]:
def bl(w_prior, sigma_prior, p, q, omega=None, delta=2.5, tau=0.02):
    """
    Compute posterior expected returns and covariances using the
    Black–Litterman reference model.

    Inputs:
        w_prior: Prior (market/benchmark) weights (N x 1), Series
        sigma_prior: Prior covariance matrix (N x N), DataFrame
        p: Pick/projection matrix (K x N), DataFrame
        q: Views vector (K x 1), Series
        omega: View-uncertainty matrix (K x K), DataFrame or None
               If None, set Omega proportional to prior variance
        delta: Risk aversion (scalar)
        tau: Prior uncertainty scaling (scalar)

    Returns:
        mu_bl: Posterior expected returns (N x 1), Series
        sigma_bl: Posterior covariance matrix (N x N), DataFrame
    """
    if omega is None:
        omega = proportional_prior(sigma_prior, tau, p)

    # Reverse-engineer equilibrium (implied) returns
    pi = implied_returns(delta, sigma_prior, w_prior)  # Series (N,)

    # Scale the prior covariance
    sigma_prior_scaled = tau * sigma_prior  # DataFrame (N x N)

    # Convenience terms
    # Invert view-space covariance (prior-projected uncertainty + view uncertainty)
    view_cov_inv = inv(p @ sigma_prior_scaled @ p.T + omega)  # (K x K)
    # View “surprise”: how far the stated views Q are from prior-implied views Pπ
    view_diff = q - (p @ pi).values  # Series (K,)

    # Posterior mean (no Omega^{-1} form)
    mu_bl = pi + (sigma_prior_scaled @ p.T @ (view_cov_inv @ view_diff))

    # Posterior covariance
    sigma_bl = sigma_prior + sigma_prior_scaled - sigma_prior_scaled @ p.T @ view_cov_inv @ p @ sigma_prior_scaled

    return mu_bl, sigma_bl


\begin{equation}
\Sigma^{BL} = \Sigma + \tau \Sigma - \tau \Sigma P^T\left(P \tau \Sigma P^T + \Omega\right)^{-1} P \tau \Sigma
\end{equation}

### A Simple Example: Absolute Views

We start with a simple two-asset example, adapted from *Statistical Models and Methods for Financial Markets* (Tze Lai and Haipeng Xing, 2008).

Consider a portfolio with two stocks: Intel (INTC) and Pfizer (PFE). From Table 3.1 (p. 72), the covariance matrix (multiplied by $10^4$) is:

\begin{array}{lcc}
& \text{INTC} & \text{PFE} \\
\text{INTC} & 46.0 & 1.06 \\
\text{PFE}  & 1.06 & 5.33
\end{array}

Assume Intel has a market capitalization of roughly $\$80B$ and Pfizer roughly $\$100B$ (not exact, but sufficient for illustration). A market-cap weighted portfolio would therefore hold:

\begin{equation}
W_{\text{INTC}} = \frac{80}{180} \approx 0.44,
\qquad
W_{\text{PFE}} = \frac{100}{180} \approx 0.56
\end{equation}

These weights are reasonable—neither stock dominates the allocation, although Pfizer is slightly overweighted.

We can now compute the equilibrium implied returns $\pi$ as follows:


In [8]:
tickers = ['INTC', 'PFE']
s = pd.DataFrame([
    [46.0, 1.06], 
    [1.06, 5.33]
], index=tickers, columns=tickers) * 10E-4
pi = implied_returns(delta=2.5, sigma=s, w=pd.Series([.44, .56], index=tickers))
pi

INTC    0.052084
PFE     0.008628
Name: Implied Returns, dtype: float64

Thus, the equilibrium implied returns are a bit above $5%$ for INTC and a bit below $1%$ for PFE.

Now suppose the investor has the following absolute views: Intel will return $2\%$, and Pfizer will rebound and return $4\%$. We can examine what Markowitz optimization would do with these inputs.

In the unconstrained case, the **maximum Sharpe ratio (MSR)** portfolio has a closed-form solution:

\begin{equation}
W_{MSR} = \frac{\Sigma^{-1}\mu_e}{\mathbf{1}^T \Sigma^{-1}\mu_e}
\end{equation}

where $\mu_e$ is the vector of expected *excess* returns and $\Sigma$ is the variance–covariance matrix.

This is implemented as follows:


In [9]:
# For convenience/readability, define the inverse of a DataFrame
def inverse(d):
    """
    Invert a square DataFrame (matrix inverse of the underlying values).
    """
    return pd.DataFrame(inv(d.values), index=d.index, columns=d.columns)

def w_msr(sigma, mu, scale=True):
    """
    Unconstrained (tangent / max Sharpe ratio) portfolio weights.

    Inputs:
        sigma: Covariance matrix (N x N), DataFrame
        mu: Expected excess returns (N x 1), Series
        scale: If True, normalize weights to sum to 1

    Returns:
        w: Portfolio weights (N x 1), Series

    Implements:
        W_MSR = (Sigma^{-1} mu) / (1^T Sigma^{-1} mu)
        (Campbell, Lo, and MacKinlay, Eq. 5.2.28)
    """
    w = inverse(sigma) @ mu
    if scale:
        w = w / w.sum()
    return w

Recall the investor’s absolute views: Intel is expected to return $2\%$ and Pfizer $4\%$. We can now compute the portfolio weights produced by a naive Markowitz (max Sharpe ratio) optimization using these expected returns directly.


In [10]:
mu_exp = pd.Series([.02, .04],index=tickers) # INTC and PFE
np.round(w_msr(s, mu_exp)*100, 2)

INTC     3.41
PFE     96.59
dtype: float64

Consistent with the well-known instability of naive Markowitz optimization, the procedure allocates an extreme weight of more than $96\%$ to Pfizer and less than $4\%$ to Intel. This is impractical—minor changes in inputs can lead to dramatically different portfolios.

Now compare this with Black–Litterman. Using the default construction for $\Omega$ and keeping the other parameters at their default values, we compute the Black–Litterman posterior inputs and the resulting MSR weights as follows:


In [11]:
# Absolute view 1: INTC will return 2%
# Absolute view 2: PFE will return 4%
q = pd.Series({"INTC": 0.02, "PFE": 0.04})

# Pick matrix (each row encodes one view)
p = pd.DataFrame(
    [
        {"INTC": 1, "PFE": 0},  # View 1: INTC
        {"INTC": 0, "PFE": 1},  # View 2: PFE
    ]
)

# Compute Black–Litterman posterior expected returns and covariance
bl_mu, bl_sigma = bl(
    w_prior=pd.Series({"INTC": 0.44, "PFE": 0.56}),
    sigma_prior=s,
    p=p,
    q=q,
)

# Black–Litterman posterior expected returns
bl_mu

INTC    0.037622
PFE     0.024111
dtype: float64

The posterior returns produced by Black–Litterman lie between the equilibrium implied returns (roughly $5\%$ for INTC and $1\%$ for PFE) and the investor’s views ($2\%$ and $4\%$). The key question is whether these blended inputs lead to more realistic portfolios. To test that, we feed the Black–Litterman expected returns and covariance matrix into the optimizer:


In [12]:
# Use the Black Litterman expected returns to get the Optimal Markowitz weights
round(w_msr(bl_sigma, bl_mu) * 100, 2)

INTC    14.07
PFE     85.93
dtype: float64

We obtain much more reasonable weights than with naive Markowitz optimization. The allocation stays close to the roughly 45/55 split of the cap-weighted portfolio, rather than jumping to extreme positions.

At the same time, the portfolio still reflects the investor’s view that Pfizer will rebound: Pfizer receives a higher weight than in the cap-weighted benchmark, but the tilt is controlled rather than drastic.


### A Simple Example: Relative Views

Next, we consider **relative** views using the same two-stock setup. Recall that the cap-weighted portfolio implies the following equilibrium expected returns:


In [13]:
# Expected returns inferred from the cap-weights
pi

INTC    0.052084
PFE     0.008628
Name: Implied Returns, dtype: float64

Recall that the cap-weighted benchmark is roughly a 45/55 mix of Intel and Pfizer.

Now suppose the investor has a **relative** view: Intel will outperform Pfizer by $2\%$. This view can be encoded in the pick matrix $P$ and view vector $Q$ as follows:


In [14]:
# Relative View 1: INTC will outperform PFE by 2%
q = pd.Series([0.02])

# Pick matrix (each row encodes one view)
p = pd.DataFrame(
    [
        {'INTC': +1, 'PFE': -1} # View 1: INTC outperforming PFE
    ]
)

# Find the Black Litterman Expected Returns
bl_mu, bl_sigma = bl(
    w_prior=pd.Series({'INTC': .44, 'PFE': .56}), 
    sigma_prior=s, 
    p=p, 
    q=q
)

# Black Litterman Implied Mu
bl_mu

INTC    0.041374
PFE     0.009646
dtype: float64

Once again, the Black–Litterman posterior expected returns fall between the cap-weight implied equilibrium returns and the investor’s view. The implied outperformance of Intel relative to Pfizer is:


In [15]:
round((pi['INTC'] - pi['PFE']) * 100, 2)

np.float64(4.35)

By assumption, the investor expected the outperformance to be only $2\%$. The Black–Litterman posterior expected returns imply a return spread that sits between the cap-weight implied spread and the investor’s stated view, reflecting a blended (shrunken) adjustment rather than a full overwrite of the prior.

In [16]:
round((bl_mu['INTC'] - bl_mu['PFE']) * 100, 2)

np.float64(3.17)

And the optimized portfolio weights obtained when we use these Black–Litterman expected returns (and the corresponding covariance estimate) are:

In [17]:
# Use the Black Litterman expected returns to get the Optimal Markowitz weights
round(w_msr(bl_sigma, bl_mu) * 100, 2)

INTC    34.72
PFE     65.28
dtype: float64

These weights are reasonable and illustrate why Black–Litterman is useful: it incorporates views while keeping the portfolio anchored near the market benchmark.

For comparison, suppose we impose the same relative view **without** Black–Litterman by directly setting expected returns to $3\%$ for Intel and $1\%$ for Pfizer (a 2% spread) and then running the optimizer.


In [18]:
round(w_msr(s, [.03, .01]) * 100, 2)

INTC    25.85
PFE     74.15
dtype: float64

The resulting weights are far more aggressive than most investors would be comfortable implementing, and likely unjustified given that the view itself is relatively mild.

In fact, if we encode the same 2% spread more pessimistically—by setting Intel to return $2\%$ and Pfizer to return $0\%$—the optimized weights become even more extreme:


In [19]:
round(w_msr(s, [.02, .0]) * 100, 2)

INTC    124.82
PFE     -24.82
dtype: float64

In this case, the naive Markowitz optimizer recommends shorting Pfizer by nearly $25\%$ of the portfolio and levering Intel to about $125\%$. That is clearly not a plausible allocation given the modest view we started with, and it highlights how unconstrained mean–variance optimization can amplify small input differences into extreme positions.


## Reproducing the He–Litterman (1999) Results

Next, we reproduce the results from \cite{he1999intuition}, which lays out the Black–Litterman procedure in a concrete, step-by-step example. The inputs below were manually transcribed from the tables in the paper and are used here to validate the implementation.

The He–Litterman example is a global allocation problem across seven countries. The data is as follows:


In [20]:
# The 7 countries
countries  = ['AU', 'CA', 'FR', 'DE', 'JP', 'UK', 'US'] 
# Table 1 of the He-Litterman paper: Correlation Matrix
rho = pd.DataFrame([
    [1.000,0.488,0.478,0.515,0.439,0.512,0.491],
    [0.488,1.000,0.664,0.655,0.310,0.608,0.779],
    [0.478,0.664,1.000,0.861,0.355,0.783,0.668],
    [0.515,0.655,0.861,1.000,0.354,0.777,0.653],
    [0.439,0.310,0.355,0.354,1.000,0.405,0.306],
    [0.512,0.608,0.783,0.777,0.405,1.000,0.652],
    [0.491,0.779,0.668,0.653,0.306,0.652,1.000]
], index=countries, columns=countries)

# Table 2 of the He-Litterman paper: volatilities
vols = pd.DataFrame(
    [0.160,0.203,0.248,0.271,0.210,0.200,0.187],
    index=countries, columns=["vol"]) 
# Table 2 of the He-Litterman paper: cap-weights
w_eq = pd.DataFrame(
    [0.016,0.022,0.052,0.055,0.116,0.124,0.615],
    index=countries, columns=["CapWeight"])
# Compute the Covariance Matrix
sigma_prior = vols.dot(vols.T) * rho
# Compute Pi and compare:
pi = implied_returns(delta=2.5, sigma=sigma_prior, w=w_eq)
(pi*100).round(1)

AU    3.9
CA    6.9
FR    8.4
DE    9.0
JP    4.3
UK    6.8
US    7.6
Name: Implied Returns, dtype: float64

The values of $\pi$ computed by the Python code exactly matches column 3 of Table 2

### View 1: Germany vs Rest of Europe

Next, we impose the view that German equities will outperform the rest of European equities by $5\%$ and **analyze what the *naive* maximum Sharpe ratio (Markowitz) optimizer does when we plug this view directly into expected returns (i.e., without using Black–Litterman)**. We examine the implied expected returns, the resulting MSR weights, and how far those weights deviate from the equilibrium portfolio.


In [21]:
r_fr = -0.008 + pi.loc["FR"]
r_uk = -0.008 + pi.loc["UK"]
r_de = 0.024 + pi.loc["DE"]

exp_r = pi.copy() 
exp_r.loc[['FR', 'DE', 'UK']] = [r_fr, r_de, r_uk]
round(w_msr(2.5 * sigma_prior, exp_r, scale = False) * 100, 2)

AU    -4.86
CA    -2.09
FR   -50.09
DE    83.29
JP    14.97
UK   -23.25
US    66.92
dtype: float64

In [22]:
exp_r - pi

AU    0.000
CA    0.000
FR   -0.008
DE    0.024
JP    0.000
UK   -0.008
US    0.000
Name: Implied Returns, dtype: float64

In [23]:
round((w_msr(2.5 * sigma_prior, exp_r, scale = False) - w_eq["CapWeight"]) * 100, 2)

AU    -6.46
CA    -4.29
FR   -55.29
DE    77.79
JP     3.37
UK   -35.65
US     5.42
dtype: float64

Although our results do not exactly match Table 3 in \cite{he1999intuition} (likely due to delta difference in the unconstrained Markowitz/MSR optimizer), the qualitative behavior is the same: the naive approach produces extreme weight shifts relative to the equilibrium portfolio.


Next, we incorporate the same view using the **Black–Litterman** framework. We express “Germany outperforming the rest of Europe” by allocating the underperformance side across France and the UK in proportion to their market capitalizations.


In [24]:
# Germany will outperform the rest of European equities (FR and UK) by 5%
q = pd.Series([0.05])  # one view

# Start with a single view: all zeros, then set the relevant entries
p = pd.DataFrame([0.0] * len(countries), index=countries).T  # shape: (1 x N)

# Split the "rest of Europe" leg across FR and UK in proportion to market caps
w_fr = w_eq.loc["FR"] / (w_eq.loc["FR"] + w_eq.loc["UK"])
w_uk = w_eq.loc["UK"] / (w_eq.loc["FR"] + w_eq.loc["UK"])

p.loc[0, "DE"] = 1.0
p.loc[0, "FR"] = -w_fr.iloc[0]
p.loc[0, "UK"] = -w_uk.iloc[0]

(p * 100).round(1)

,AU,CA,FR,DE,JP,UK,US
0,0.0,0.0,-29.5,100.0,0.0,-70.5,0.0


The results of implementing this view are reported in Table 4 of \cite{he1999intuition}. Our implementation reproduces **Column 1** of Table 4 exactly. Next, we examine the corresponding Black–Litterman posterior expected returns, $\mu^{BL}$:


In [25]:
delta = 2.5
tau = 0.05 # from Footnote 8
# Find the Black Litterman Expected Returns
bl_mu, bl_sigma = bl(w_eq, sigma_prior, p, q, tau = tau)
(bl_mu*100).round(1)

AU     4.3
CA     7.6
FR     9.3
DE    11.0
JP     4.5
UK     7.0
US     8.1
dtype: float64

The Black–Litterman expected returns produced by the code match **Column 2** of Table 4 exactly.

He and Litterman then compute the optimal portfolio $w^{*}$ using the following expression (Equation (13) on page 6 of \cite{he1999intuition}):


In [26]:
def w_star(delta, sigma, mu):
    return (inverse(sigma).dot(mu))/delta

wstar = w_star(delta=2.5, sigma=bl_sigma, mu=bl_mu)

(wstar*100).round(1)

AU     1.5
CA     2.1
FR    -4.0
DE    35.4
JP    11.0
UK    -9.5
US    58.6
dtype: float64

The computed $w^{*}$ matches **Column 3** ($w^{*}$) of Table 4 exactly. Finally, they report the weight shift relative to equilibrium in Column 4 by computing:

\begin{equation}
w^{*} - \frac{w_{eq}}{1+\tau}
\end{equation}

i.e., the difference between the optimal portfolio and the (unscaled) equilibrium portfolio weights. We replicate that column as follows:


In [27]:
w_eq  = w_msr(delta*sigma_prior, pi, scale=False)

np.round(wstar - w_eq/(1+tau), 3)*100

AU    -0.0
CA    -0.0
FR    -8.9
DE    30.2
JP     0.0
UK   -21.3
US     0.0
dtype: float64

This exactly matches **Column 4** of Table 4, completing our reproduction of the first-view example in \cite{he1999intuition}.

This highlights the appeal of the Black–Litterman approach: assets not involved in the view remain essentially unchanged, while the countries implicated by the view adjust in the expected direction—France and the UK are underweighted and Germany is overweighted—but the shifts are controlled rather than extreme, unlike what a naive optimizer would typically produce.


### View 2: Canada vs US

In the second case, He and Litterman add a new relative view: **Canadian equities will outperform US equities by $3\%$**. The corresponding results are reported in Table 5 of \cite{he1999intuition}, which we reproduce next.


In [28]:
view2 = pd.Series([.03], index=[1])
q = pd.concat([q, view2])
pick2 = pd.DataFrame([0.]*len(countries), index=countries, columns=[1]).T
p = pd.concat([p, pick2])
p.loc[1, 'CA']=+1
p.loc[1, 'US']=-1
np.round(p.T, 3)*100

,0,1
AU,0.0,0.0
CA,0.0,100.0
FR,-29.5,0.0
DE,100.0,0.0
JP,0.0,0.0
UK,-70.5,0.0
US,0.0,-100.0


This matches Columns 1 and 2 of Table 5. We now compute the Black–Litterman weights as before:

In [29]:
bl_mu, bl_sigma = bl(w_eq, sigma_prior, p, q, tau = tau)
np.round(bl_mu*100, 1)

AU     4.4
CA     8.7
FR     9.5
DE    11.2
JP     4.6
UK     7.0
US     7.5
dtype: float64

The Black–Litterman expected returns computed by the Python code exactly reproduce Column 3 of Table 5.

He and Litterman compute the optimal portfolio $w^{*}$ as follows (Equation (13) on page 6 of \cite{he1999intuition}):


In [30]:
wstar = w_star(delta=2.5, sigma=bl_sigma, mu=bl_mu)

(wstar*100).round(1)

AU     1.5
CA    41.9
FR    -3.4
DE    33.6
JP    11.0
UK    -8.2
US    18.8
dtype: float64

The computed $w^{*}$ exactly replicates Column 4 ($w^{*}$) of Table 5. Finally, as in the previous case, they report the weight shift in Column 5 by computing:

\begin{equation}
w^{*} - \frac{w_{eq}}{1+\tau}
\end{equation}

We replicate that column as follows:


In [31]:
w_eq  = w_msr(delta*sigma_prior, pi, scale=False)

np.round(wstar - w_eq/(1+tau), 3)*100

AU    -0.0
CA    39.8
FR    -8.4
DE    28.4
JP    -0.0
UK   -20.0
US   -39.8
dtype: float64

Which exactly reproduces the last column of Table 5.

Once again, this shows the benefit of the approach. Assets not involved in the view (AU, JP) remain unchanged. Countries implied to underperform (FR, UK, US) are underweighted, while those implied to outperform (CA, DE) are overweighted—without the extreme allocations a naive optimizer would typically produce.


### View 3: More Bullish Canada vs US

In the third case, He and Litterman strengthen the second view by increasing Canada’s expected outperformance versus the US from $3%$ to $4%$. The corresponding results appear in Table 6, which we reproduce next.


In [32]:
q[1] = .04
q

0    0.05
1    0.04
dtype: float64

Note that $P$ remains unchanged because we only changed $Q$ (the view magnitude), not the assets involved in the view.


In [33]:
round(p.T * 100, 1)

,0,1
AU,0.0,0.0
CA,0.0,100.0
FR,-29.5,0.0
DE,100.0,0.0
JP,0.0,0.0
UK,-70.5,0.0
US,0.0,-100.0


This matches Columns 1 and 2 of Table 6. We now compute the Black–Litterman weights as before:

In [34]:
bl_mu, bl_sigma = bl(w_eq, sigma_prior, p, q, tau = tau)
np.round(bl_mu*100, 1)

AU     4.4
CA     9.1
FR     9.5
DE    11.3
JP     4.6
UK     7.0
US     7.3
dtype: float64

The Black–Litterman expected returns computed by the Python code exactly reproduce Column 3 of Table 6.

He and Litterman compute the optimal portfolio $w^{*}$ as follows (Equation (13) on page 6 of \cite{he1999intuition}):

In [35]:
wstar = w_star(delta=2.5, sigma=bl_sigma, mu=bl_mu)

(wstar*100).round(1)

AU     1.5
CA    53.3
FR    -3.3
DE    33.1
JP    11.0
UK    -7.8
US     7.3
dtype: float64

The computed $w^{*}$ exactly replicates Column 4 ($w^{*}$) of Table 6. Finally, as in the previous case, they report the weight shift in Column 5 by computing:

\begin{equation}
w^{*} - \frac{w_{eq}}{1+\tau}
\end{equation}

We replicate that column as follows:

In [36]:
w_eq  = w_msr(delta*sigma_prior, pi, scale=False)

np.round(wstar - w_eq/(1+tau), 3)*100

AU    -0.0
CA    51.3
FR    -8.2
DE    27.8
JP    -0.0
UK   -19.6
US   -51.3
dtype: float64

Which exactly reproduces the last column of Table 6. Again, the weights shift further in the direction of the stronger view, while still avoiding the extreme allocations that a naive optimizer would typically produce.


### View 4: Increasing View Uncertainty

As a final step, He and Litterman illustrate the role of $\Omega$. They increase the uncertainty associated with the first view (Germany outperforming the rest of Europe). We first compute the default $\Omega$, then increase only the uncertainty of that first view.


In [37]:
# Default "proportional to prior" Omega
omega = proportional_prior(sigma_prior, tau, p)

# Double the uncertainty associated with View 1
omega.iloc[0, 0] = 2 * omega.iloc[0, 0]

np.round(p.T * 100, 1)

,0,1
AU,0.0,0.0
CA,0.0,100.0
FR,-29.5,0.0
DE,100.0,0.0
JP,0.0,0.0
UK,-70.5,0.0
US,0.0,-100.0


This matches Columns 1 and 2 of Table 7 (as expected, since we only changed $\Omega$, not $Q$ or $P$). We now compute the Black–Litterman weights as before, but pass in the adjusted $\Omega$:


In [38]:
bl_mu, bl_sigma = bl(w_eq, sigma_prior, p, q, omega=omega, tau=tau)
np.round(bl_mu, 3)*100

AU     4.3
CA     8.9
FR     9.3
DE    10.6
JP     4.6
UK     6.9
US     7.2
dtype: float64

The Black–Litterman expected returns computed by the code exactly reproduce Column 3 of Table 7.

He and Litterman compute the optimal portfolio $w^{*}$ as follows (Equation (13) on page 6 of \cite{he1999intuition}):


In [39]:
wstar = w_star(delta=2.5, sigma=bl_sigma, mu=bl_mu)

(wstar*100).round(1)

AU     1.5
CA    53.9
FR    -0.5
DE    23.6
JP    11.0
UK    -1.1
US     6.8
dtype: float64

The computed $w^{*}$ exactly replicates Column 4 ($w^{*}$) of Table 7. Finally, they report the weight shift in Column 6 by computing:

\begin{equation}
w^{*} - \frac{w_{eq}}{1+\tau}
\end{equation}

We replicate that column as follows:


In [40]:
w_eq  = w_msr(delta*sigma_prior, pi, scale=False)

np.round(wstar - w_eq/(1+tau), 3)*100

AU    -0.0
CA    51.8
FR    -5.4
DE    18.4
JP    -0.0
UK   -13.0
US   -51.8
dtype: float64

In [41]:
A = omega / tau + p @ sigma_prior / (1+tau) @ p.T
lamda = tau * inv(omega) @ q / delta  - inv(A) @ p @ sigma_prior / (1+tau) @w_eq - inv(A) @ p @ sigma_prior / (1+tau) @ p.T @(tau * inv(omega) @ q / delta)
lamda

0    0.193109
1    0.543851
dtype: float64

Which exactly reproduces the last column of Table 7. Again, the weights move further in the direction implied by the views, while still avoiding the extreme allocations that a naive optimizer would tend to produce.

That concludes our reproduction of the paper. He and Litterman also include an additional table (Table 8) illustrating the effect of adding a third view. However, that third view is identical to the equilibrium-implied values, so the results are unchanged from Table 7. For that reason, I do not reproduce Table 8 here.


In [42]:
imp_exp_r = delta * sigma_prior @ wstar

In [43]:
q[3] = imp_exp_r['CA'] - imp_exp_r['JP']
q

0    0.050000
1    0.040000
3    0.041186
dtype: float64

In [44]:
p

,AU,CA,FR,DE,JP,UK,US
0,0.0,0.0,-0.295455,1.0,0.0,-0.704545,0.0
1,0.0,1.0,0.000000,0.0,0.0,0.000000,-1.0


In [45]:
pick3 = pd.DataFrame([0.]*len(countries), index=countries, columns=[2]).T
p = pd.concat([p, pick3])
p.loc[2, 'CA']=+1
p.loc[2, 'JP']=-1
np.round(p.T, 3)*100

,0,1,2
AU,0.0,0.0,0.0
CA,0.0,100.0,100.0
FR,-29.5,0.0,0.0
DE,100.0,0.0,0.0
JP,0.0,0.0,-100.0
UK,-70.5,0.0,0.0
US,0.0,-100.0,0.0


In [46]:
omega = proportional_prior(sigma_prior, tau, p)

# Double the uncertainty associated with View 1
omega.iloc[0, 0] = 2 * omega.iloc[0, 0]

In [47]:
bl_mu, bl_sigma = bl(w_eq, sigma_prior, p, q, omega=omega, tau=tau)
np.round(bl_mu, 3)*100

AU     4.3
CA     8.8
FR     9.2
DE    10.6
JP     4.6
UK     6.9
US     7.1
dtype: float64

In [48]:
np.diag(omega / tau)

array([0.04259737, 0.01703476, 0.0588784 ])

In [49]:
wstar = w_star(delta=2.5, sigma=bl_sigma, mu=bl_mu)
wstar

AU    0.015238
CA    0.538906
FR   -0.004814
DE    0.236294
JP    0.110476
UK   -0.011480
US    0.067761
dtype: float64

In [50]:
A = omega / tau + p @ sigma_prior / (1+tau) @ p.T

In [51]:
lamda = tau * inv(omega) @ q / delta  - inv(A) @ p @ sigma_prior / (1+tau) @w_eq - inv(A) @ p @ sigma_prior / (1+tau) @ p.T @(tau * inv(omega) @ q / delta)
lamda

0    1.931086e-01
1    5.438512e-01
2   -5.551115e-17
dtype: float64

## Applying on Industry Data 

Now that we have validated the implementation, we apply it to the Fama–French industry portfolios.

Start by loading the data as follows:


Load the 49-industry value-weighted returns and market-cap weights, restrict the sample to **2013–2018 (inclusive)**, and use the **starting cap weights** of that window. Focus on the five industries **Hlth, Fin, Whlsl, Rtail, Food**. Compute the **correlation matrix** and **annualized volatilities** (monthly vol $\times \sqrt{12}$), then form the prior covariance matrix using the same approach as earlier. Finally, with **$\delta = 2.5$** (He–Litterman), compute the **implied equilibrium returns** vector $\pi = \delta \Sigma w$.


In [52]:
import risk_kit as rk

ind49_rets = rk.get_ind_returns(weighting="vw", n_inds=49)["2013":]
ind49_mcap = rk.get_ind_market_caps(49, weights=True)["2013":]
inds = ['Hlth', 'Fin', 'Whlsl', 'Rtail', 'Food']
rho_ = ind49_rets[inds].corr()
vols_ = (ind49_rets[inds].std()*np.sqrt(12))    
sigma_prior_ =  (vols_.T).dot(vols_) * rho_

In [53]:
sigma_prior_

,Hlth,Fin,Whlsl,Rtail,Food
Hlth,0.108718,0.057064,0.070764,0.062429,0.040410
Fin,0.057064,0.108718,0.084545,0.064723,0.037292
Whlsl,0.070764,0.084545,0.108718,0.080859,0.058843
Rtail,0.062429,0.064723,0.080859,0.108718,0.062519
Food,0.040410,0.037292,0.058843,0.062519,0.108718


Normalize the cap weights within the five-industry subset (each industry’s cap weight divided by the sum across the five). The industry with the highest normalized cap weight is:


In [54]:
w_eq = ind49_mcap.iloc[0][inds] / ind49_mcap.iloc[0][inds].sum() 
w_eq.idxmax()

'Rtail'

Identify the industry with the highest implied (equilibrium) return $\pi$ among the five-sector subset.


In [55]:
pi = implied_returns(delta, sigma_prior_, w_eq)
pi.idxmax()

'Rtail'

Identify the industry with the lowest implied (equilibrium) return $\pi$ among the five-sector subset.

In [56]:
pi.idxmin()

'Hlth'

Impose the relative view that **Hlth will outperform a market-cap-weighted blend of Rtail and Whlsl by 3%**, and encode this as a single row in the pick matrix (P) (long Hlth, short Rtail/Whlsl split by their market-cap weights).


In [57]:
# Hlth will outperform Rtail and Whlsl by 3%
q_ = pd.Series([0.03])  # one view

# Start with a single view: all zeros, then set the relevant entries
p_ = pd.DataFrame([0.0] * len(inds), index=inds).T  # shape: (1 x N)

# Split Rtail and Whlsl in proportion to market caps
w_rtail = w_eq.loc["Rtail"] / (w_eq.loc["Rtail"] + w_eq.loc["Whlsl"])
w_whlsl = w_eq.loc["Whlsl"] / (w_eq.loc["Rtail"] + w_eq.loc["Whlsl"])

p_.loc[0, "Hlth"] = 1.0
p_.loc[0, "Rtail"] = -w_rtail
p_.loc[0, "Whlsl"] = -w_whlsl

(p_).round(2)

,Hlth,Fin,Whlsl,Rtail,Food
0,1.0,0.0,-0.15,-0.85,0.0


After imposing this view under Black–Litterman (using $\delta=2.5$ and $\tau=0.05$), identify which of the five sectors has the **lowest posterior expected return** $\mu^{BL}$.


In [58]:
bl_mu, bl_sigma = bl(w_eq, sigma_prior_, p_, q_, omega=None, delta=2.5, tau=0.05)

In [59]:
bl_mu.idxmin()

'Food'

Identify which sector has the **highest weight** in the unconstrained MSR (tangent) portfolio when optimized using the Black–Litterman posterior inputs ($\mu^{BL}, \Sigma^{BL}$).


In [60]:
w_msr_ = w_msr(bl_sigma, bl_mu, scale=False)

In [61]:
w_msr_.idxmax()

'Rtail'

Identify which sector has the **lowest weight** in the unconstrained MSR (tangent) portfolio when optimized using the Black–Litterman posterior inputs ($\mu^{BL}, \Sigma^{BL}$).


In [62]:
w_msr_.idxmin()

'Whlsl'

Update the relative view magnitude: keep the same (P) (same assets and cap-weight split), but increase the outperformance from **3% to 5%** by updating (Q) only.


In [63]:
# Hlth will outperform Rtail and Whlsl by 5%
q_ = pd.Series([0.05])  # one view

bl_mu, bl_sigma = bl(w_eq, sigma_prior_, p_, q_, omega=None, delta=2.5, tau=0.05)

Under the updated view (with (Q) increased to 5%), identify which sector has the **highest** Black–Litterman posterior expected return $\mu^{BL}$.

In [64]:
bl_mu.idxmax()

'Rtail'

Under the updated view (5% outperformance), identify which sector receives the **highest weight** in the MSR portfolio optimized using the Black–Litterman posterior inputs.


In [65]:
w_msr_ = w_msr(bl_sigma, bl_mu, scale=False)
w_msr_.idxmax()

'Hlth'